# AgriSense — Minimal Stage 1 Training (PlantVillage)A deliberately small, self-contained run that produces **real artifacts** you can show:| Artifact | What it is ||---|---|| `results.csv` | Per-epoch loss + accuracy (the training log) || `results.png` | Training curves (loss / top-1 / top-5 vs epoch) || `confusion_matrix.png` | Which classes get confused with which || `args.yaml` | Every hyperparameter used, auto-recorded || `weights/best.pt` | Trained PyTorch weights || `species.onnx` | **The exported ONNX model** || `metrics.json` | Final top-1 / top-5 on the validation split |This bypasses the full 4-dataset pipeline in `ml/` (which needs ~13 GB and Drive).It trains **PlantVillage only, few epochs** — enough for real numbers, notstate-of-the-art. That is the honest framing: a working baseline, not a finished model.**Runtime → Change runtime type → T4 GPU** before running anything.

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Install Ultralytics

In [ ]:
%pip install -q ultralytics onnxruntimeimport ultralyticsultralytics.checks()

## 3. Get PlantVillage from KaggleYou need a Kaggle API token: kaggle.com → your profile → Settings → API →**Create New Token**. That downloads `kaggle.json`. Run the cell and upload it.

In [ ]:
from google.colab import filesimport os, pathlibif not pathlib.Path('/root/.kaggle/kaggle.json').exists():    print('Upload your kaggle.json now:')    files.upload()    os.makedirs('/root/.kaggle', exist_ok=True)    os.rename('kaggle.json', '/root/.kaggle/kaggle.json')    os.chmod('/root/.kaggle/kaggle.json', 0o600)print('kaggle.json in place.')

In [ ]:
# Same slug the real pipeline uses (ml/configs/sources.yaml -> plantvillage)!kaggle datasets download -d mohitsingh1804/plantvillage -p /content/dl --unzipprint('done')

## 4. Look at what actually downloadedNever assume the folder layout — print it and adapt. This is the step thatcatches a mirror having a different structure than expected.

In [ ]:
import pathlibROOT = pathlib.Path('/content/dl')def tree(p, depth=0, max_depth=3, max_children=6):    if depth > max_depth:        return    kids = sorted([c for c in p.iterdir()])[:max_children]    for c in kids:        n = len(list(c.iterdir())) if c.is_dir() else ''        print('  ' * depth + f'{c.name}{"/" if c.is_dir() else ""}  {n}')        if c.is_dir():            tree(c, depth + 1, max_depth, max_children)tree(ROOT)

## 5. Build a train/val splitUltralytics classification wants `dataset/train/<class>/*.jpg` and`dataset/val/<class>/*.jpg`. This cell handles both cases: an existingtrain/val split, or a flat folder-per-class that needs splitting.`MAX_PER_CLASS` caps images per class — lower it to make the run faster.

In [ ]:
import pathlib, random, shutil, collectionsrandom.seed(42)                 # same seed as ml/configs/paths.yamlMAX_PER_CLASS = 400             # None = use everything. 400 keeps a run ~10 min.VAL_FRACTION  = 0.2SRC = pathlib.Path('/content/dl')DST = pathlib.Path('/content/pv_split')def find_class_root(root):    """Find the directory whose children are class folders full of images."""    best, best_n = None, 0    for d in [root] + [p for p in root.rglob('*') if p.is_dir()]:        subs = [s for s in d.iterdir() if s.is_dir()]        if len(subs) < 5:            continue        imgs = sum(len(list(s.glob('*.jp*g'))) + len(list(s.glob('*.png'))) for s in subs[:5])        if imgs > best_n:            best, best_n = d, imgs    return best# If the download already has train/ and val(id)/, reuse it as-is.existing = Nonefor d in [SRC] + [p for p in SRC.rglob('*') if p.is_dir()]:    names = {s.name.lower() for s in d.iterdir() if s.is_dir()}    if 'train' in names and ({'val', 'valid', 'validation'} & names):        existing = d        breakif existing:    val_name = next(s.name for s in existing.iterdir() if s.is_dir() and s.name.lower() in ('val','valid','validation'))    print(f'Found existing split at {existing} (train/ + {val_name}/) — using it directly.')    if DST.exists():        shutil.rmtree(DST)    DST.mkdir(parents=True)    (DST / 'train').symlink_to(existing / 'train', target_is_directory=True)    (DST / 'val').symlink_to(existing / val_name, target_is_directory=True)else:    croot = find_class_root(SRC)    print(f'Flat layout detected at {croot} — building an 80/20 split.')    if DST.exists():        shutil.rmtree(DST)    classes = sorted([c for c in croot.iterdir() if c.is_dir()])    for c in classes:        imgs = sorted(list(c.glob('*.jp*g')) + list(c.glob('*.png')) + list(c.glob('*.JPG')))        random.shuffle(imgs)        if MAX_PER_CLASS:            imgs = imgs[:MAX_PER_CLASS]        n_val = max(1, int(len(imgs) * VAL_FRACTION))        for split, chunk in (('val', imgs[:n_val]), ('train', imgs[n_val:])):            out = DST / split / c.name            out.mkdir(parents=True, exist_ok=True)            for img in chunk:                shutil.copy2(img, out / img.name)counts = {}for split in ('train', 'val'):    cs = sorted([d for d in (DST / split).iterdir() if d.is_dir()])    counts[split] = sum(len(list(d.iterdir())) for d in cs)    print(f'{split}: {len(cs)} classes, {counts[split]} images')

## 6. Train`yolo11n-cls` (nano) instead of the `yolo11s-cls` in `ml/configs/train_stage1.yaml`,and 5 epochs instead of 30 — that is the whole "minimal" change. Everything else(224 px, seed 42, cosine LR) matches the real config.Ultralytics writes the log, curves and confusion matrix automatically.

In [ ]:
from ultralytics import YOLOEPOCHS = 5          # bump to 10-15 if you have time; 5 is enough for real curvesIMGSZ  = 224        # matches ml/configs/train_stage1.yamlBATCH  = 64model = YOLO('yolo11n-cls.pt')results = model.train(    data=str(DST),    imgsz=IMGSZ,    epochs=EPOCHS,    batch=BATCH,    seed=42,    cos_lr=True,    project='/content/runs/stage1',    name='species',    exist_ok=True,    plots=True,          # <- this is what writes results.png + confusion_matrix.png)RUN_DIR = pathlib.Path(model.trainer.save_dir)print('Run directory:', RUN_DIR)

## 7. Validate and record final metrics

In [ ]:
import jsonmetrics = model.val(data=str(DST), split='val')top1 = float(metrics.top1)top5 = float(metrics.top5)class_names = [model.names[i] for i in sorted(model.names.keys())]print(f'top-1 accuracy: {top1:.4f}')print(f'top-5 accuracy: {top5:.4f}')print(f'classes: {len(class_names)}')summary = {    'model': 'yolo11n-cls',    'epochs': EPOCHS,    'imgsz': IMGSZ,    'batch': BATCH,    'seed': 42,    'dataset': 'PlantVillage (mohitsingh1804/plantvillage)',    'train_images': counts['train'],    'val_images': counts['val'],    'num_classes': len(class_names),    'classes': class_names,    'val_top1': top1,    'val_top5': top5,}with open(RUN_DIR / 'metrics.json', 'w') as f:    json.dump(summary, f, indent=2)print('wrote', RUN_DIR / 'metrics.json')

## 8. Show the graphs inlineThese render in the notebook so they are visible in the saved `.ipynb` too.

In [ ]:
from IPython.display import Image, displayimport pandas as pdprint('=== Training log (results.csv) ===')display(pd.read_csv(RUN_DIR / 'results.csv'))for png in ['results.png', 'confusion_matrix.png', 'confusion_matrix_normalized.png']:    p = RUN_DIR / png    if p.exists():        print(f'=== {png} ===')        display(Image(filename=str(p)))    else:        print(f'({png} not produced)')

## 9. Export to ONNX — and prove it actually runsSame export settings as `ml/src/agrisense_pd/export/to_onnx.py` (opset 12,dynamic, simplified, 224 px). The verification step matters: it is thedifference between "here is an .onnx file" and "here is a model I can run".

In [ ]:
onnx_path = model.export(format='onnx', opset=12, dynamic=True, simplify=True, imgsz=IMGSZ)print('exported:', onnx_path)

In [ ]:
import onnxruntime as ortimport numpy as npsess = ort.InferenceSession(str(onnx_path), providers=['CPUExecutionProvider'])inp = sess.get_inputs()[0]out = sess.get_outputs()[0]print('input :', inp.name, inp.shape, inp.type)print('output:', out.name, out.shape)# Run one real forward pass on random input to prove the graph executes.dummy = np.random.rand(1, 3, IMGSZ, IMGSZ).astype(np.float32)logits = sess.run(None, {inp.name: dummy})[0]pred = int(np.argmax(logits))print(f'forward pass OK -> logits {logits.shape}, argmax class {pred} = {class_names[pred]}')

## 10. Predict on a real imageRun the trained model on an actual validation photo — this is what to do liveif someone asks "does it actually work?".

In [ ]:
import randomfrom IPython.display import Image, displaysample_class = random.choice(sorted([d for d in (DST / 'val').iterdir() if d.is_dir()]))sample_img = random.choice(list(sample_class.iterdir()))print('True label:', sample_class.name)display(Image(filename=str(sample_img), width=250))r = model.predict(str(sample_img), imgsz=IMGSZ, verbose=False)[0]top5_idx = r.probs.top5print('\nModel prediction:')for i in top5_idx:    print(f'  {class_names[i]:45s} {float(r.probs.data[i]):.4f}')

## 11. Collect everything into one zipDownload this and keep it with the project — it is your evidence bundle.

In [ ]:
import shutil, pathlibOUT = pathlib.Path('/content/agrisense_stage1_artifacts')if OUT.exists():    shutil.rmtree(OUT)OUT.mkdir(parents=True)wanted = [    'results.csv', 'results.png', 'args.yaml', 'metrics.json',    'confusion_matrix.png', 'confusion_matrix_normalized.png',]for name in wanted:    src = RUN_DIR / name    if src.exists():        shutil.copy2(src, OUT / name)(OUT / 'weights').mkdir()for w in ['best.pt', 'last.pt', 'best.onnx']:    src = RUN_DIR / 'weights' / w    if src.exists():        shutil.copy2(src, OUT / 'weights' / ('species.onnx' if w == 'best.onnx' else w))for f in sorted(OUT.rglob('*')):    if f.is_file():        print(f'{f.relative_to(OUT)}  ({f.stat().st_size // 1024} KB)')shutil.make_archive('/content/agrisense_stage1_artifacts', 'zip', OUT)print('\nzip ready: /content/agrisense_stage1_artifacts.zip')

In [ ]:
from google.colab import filesfiles.download('/content/agrisense_stage1_artifacts.zip')

## 12. What you now have, and how to describe itEverything in the zip was produced by this run. Nothing is fabricated.**If asked "where is your model?"** — `weights/species.onnx`, exported at opset 12,verified to load and run a forward pass in onnxruntime (cell 9).**If asked "how well does it work?"** — quote `metrics.json`: top-1 and top-5 on aheld-out 20% validation split, and show `confusion_matrix.png`.**If asked "why only 5 epochs / why nano?"** — because this is a baseline proving thepipeline end to end. The production config (`ml/configs/train_stage1.yaml`) is`yolo11s-cls` for 30 epochs across four datasets; this is the same code path atreduced scale.**Be upfront about the limitation:** PlantVillage is lab imagery — single detachedleaves on plain backgrounds. High accuracy here does **not** transfer to field photos;that is exactly why `ml/configs/paths.yaml` holds out PlantDoc as a real-world eval setand why `documents/DATASETS.md` documents the deduplication against it. Saying thisyourself is far stronger than being caught by it.**To plug this into the app:** the backend already switches inference source byenvironment variable — set `PLANT_API_URL` + `PLANT_API_KEY` and `/api/plant/predict`proxies to the ONNX serving layer instead of Gemini. No code change.